# Cleaning the Data

    - imports

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('../data/Metro_Interstate_Traffic_Volume.csv')
data.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


    - Transforming holiday to binary (int) as new column is_holiday & dropping holiday

In [3]:
data['is_holiday'] = data['holiday'].notnull().astype(int)
data = data.drop('holiday', axis=1)

In [4]:
data.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,is_holiday
0,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,0
1,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,0
2,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,0
3,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,0
4,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,0


In [5]:
data['weather_main'].value_counts()

weather_main
Clouds          15164
Clear           13391
Mist             5950
Rain             5672
Snow             2876
Drizzle          1821
Haze             1360
Thunderstorm     1034
Fog               912
Smoke              20
Squall              4
Name: count, dtype: int64

In [6]:
data['weather_description'].nunique()

38

    - Dropping weather description because it creates a noise for the model

In [7]:
data = data.drop('weather_description', axis=1)

In [8]:
data['date_time'] = pd.to_datetime(data['date_time'])

In [9]:
data.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,is_holiday
0,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,0
1,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,0
2,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,0
3,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,0
4,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,0


In [10]:
data['hour'] = data['date_time'].dt.hour
data['day_of_week'] = data['date_time'].dt.dayofweek
data['month'] = data['date_time'].dt.month

In [11]:
data.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,is_holiday,hour,day_of_week,month
0,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,0,9,1,10
1,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,0,10,1,10
2,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,0,11,1,10
3,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,0,12,1,10
4,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,0,13,1,10


In [12]:
# Hour transformation
data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)

# Month transformation (optional but good for long datasets)
data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)

# Now drop the original date_time and the raw hour/month columns
data = data.drop(['date_time', 'hour', 'month'], axis=1)

    - Removing impossible rain outlier

In [13]:
# Remove the physically impossible rain outlier (> 9800mm)
# We keep only rows where rain is less than 100mm/h
data = data[data['rain_1h'] < 100]

    - Removing invalid temperature

In [14]:
# Remove the invalid 0.0 Kelvin temperature rows
# We keep only rows where temperature is greater than 0
data = data[data['temp'] > 0]

    - Removing duplicate rows

In [15]:
data = data.drop_duplicates()

In [16]:
data.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,traffic_volume,is_holiday,day_of_week,hour_sin,hour_cos,month_sin,month_cos
0,288.28,0.0,0.0,40,Clouds,5545,0,1,7.071068e-01,-0.707107,-0.866025,0.5
1,289.36,0.0,0.0,75,Clouds,4516,0,1,5.000000e-01,-0.866025,-0.866025,0.5
2,289.58,0.0,0.0,90,Clouds,4767,0,1,2.588190e-01,-0.965926,-0.866025,0.5
3,290.13,0.0,0.0,90,Clouds,5026,0,1,1.224647e-16,-1.000000,-0.866025,0.5
4,291.14,0.0,0.0,75,Clouds,4918,0,1,-2.588190e-01,-0.965926,-0.866025,0.5


In [17]:
data.to_csv('../data/traffic_data_cleaned_&_engineered.csv', index=False)